In [ ]:
import pymysql
import glob
import os
import pandas as pd 

db = pymysql.connect(
    host = 'localhost',
    user = 'root', 
    password = #enter your mySQL workbench password here 
)

dataframes = [] 
folder = #download the data from discord and place the filepath where it resides here 
csvs = glob.glob(os.path.join(folder, "*.csv"))

try:
    for items in csvs: 
        print(f"currently reading in {items}")
        df = pd.read_csv(items)
        dataframes.append(df)
    print("successfully read in files to dataframe")
except: 
    print("failed to read in the csvs to dataframe")

combined_df = pd.concat(dataframes, ignore_index=True)

combined_df

currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001031.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001101.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001102.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001103.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001104.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001105.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbaron13/nba-shots-dataset-2001-present/versions/2/nba\20001106.csv
currently reading in C:/Users/natha/.cache/kagglehub/datasets/techbar

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,match_id,shotX,shotY,quarter,time_remaining,player,team,made,shot_type,distance,score,opp,status
0,0,0,0,200010310ATL,8.3,20.2,1st quarter,11:29.0,Baron Davis,CHH,False,2-pointer,22,0-2,'CHH',trails
1,1,1,1,200010310ATL,35.9,15.8,1st quarter,10:41.0,Jamal Mashburn,CHH,False,2-pointer,16,0-2,'CHH',trails
2,2,2,2,200010310ATL,24.0,5.0,1st quarter,10:38.0,Baron Davis,CHH,False,2-pointer,0,0-2,'CHH',trails
3,3,3,3,200010310ATL,15.4,5.9,1st quarter,9:36.0,Elden Campbell,CHH,False,2-pointer,9,0-4,'CHH',trails
4,4,4,4,200010310ATL,20.3,25.0,1st quarter,9:18.0,David Wesley,CHH,True,2-pointer,20,2-4,'CHH',trails
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317694,1795,1795,1795,200202120SEA,3.6,7.6,4th quarter,1:05.0,Desmond Mason,SEA,False,2-pointer,21,102-106,'DAL',trails
317695,1796,1796,1796,200202120SEA,20.7,31.2,4th quarter,0:32.7,Rashard Lewis,SEA,False,3-pointer,26,102-108,'DAL',trails
317696,1797,1797,1797,200202120SEA,24.0,5.0,4th quarter,0:24.5,Gary Payton,SEA,True,2-pointer,0,104-108,'DAL',trails
317697,1798,1798,1798,200202120SEA,24.0,5.0,4th quarter,0:14.2,Brent Barry,SEA,True,2-pointer,0,106-110,'DAL',trails


In [23]:
made_shots = combined_df[combined_df["made"] == True]

def format_into_sql_date(s: str) -> str:
    result = "" 
    c = 0
    while c <= 7: 
        if c == 4: 
            result += "-"
        if c == 6: 
            result += "-"
        result += s[c]
        c += 1 
    return result

made_shots["match_id"] = made_shots["match_id"].apply(format_into_sql_date)                             

made_shots.rename(columns={"match_id":'date'}, inplace=True)

finalized_data = pd.DataFrame({"date": made_shots["date"], "shotX": made_shots["shotX"], "shotY": made_shots["shotY"], "team": made_shots["team"]})

C:\Users\natha\AppData\Local\Temp\ipykernel_32324\1610028410.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  made_shots["match_id"] = made_shots["match_id"].apply(format_into_sql_date)
C:\Users\natha\AppData\Local\Temp\ipykernel_32324\1610028410.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  made_shots.rename(columns={"match_id":'date'}, inplace=True)


In [29]:
db = pymysql.connect(
    host = 'localhost',
    user = 'root', 
    password = 'szm6N5zm8vqwi$' 
)

try:
    cursor = db.cursor()

    cursor.execute("create database if not exists nba_shots")
    cursor.execute("use nba_shots")

    grouped = finalized_data.groupby("team")

    for team, data in grouped: 

        table_statement = f"""
            create table if not exists {team} ( 
                date DATE,
                shotX FLOAT,
                shotY FLOAT,
                team VARCHAR(10)
            )
        """
        cursor.execute(table_statement)

        insert_statement = f"""
            insert into {team}
            (date, shotX, shotY, team)
            values(%s, %s, %s, %s)
        """

        final_entry = list(
            zip(
                data["date"],
                data["shotX"],
                data["shotY"],
                data["team"],
            )
        )
        cursor.executemany(insert_statement, final_entry)

        db.commit()
finally: 
    cursor.close() 
    db.close()